# Somo la 03 - Mifumo ya Ubunifu wa Mawakala

Katika somo hili, tunachunguza mifumo mitatu ya msingi ya ubunifu kwa ajili ya kujenga mawakala wa AI wenye ufanisi:

1. **Maelekezo Yaziwafaa kwa Wakala** — Kuandika maagizo sahihi, yanayoelezea majukumu yanayoongoza tabia ya wakala  
2. **Matokeo Yenye Muundo kwa Modeli za Pydantic** — Kuhakikisha mawakala wanarejea data inayoweza kutabirika na kuthibitishwa  
3. **Mawakala wa Majukumu Mmoja** — Kubuni mawakala wenye umakini wanaofanya kazi moja kwa ufanisi  

Tutatumia kila mfumo kwenye hali ya **mtoaji mapendekezo ya sehemu za kusafiria**, tukijenga polepole mfumo unaoweza kupendekeza maeneo, kukagua upatikanaji, na kushughulikia masuala ya usafirishaji.


## Usanidi


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Mfano 1: Maelekezo Wazi kwa Wakala

Mfano wenye athari kubwa pia ni rahisi: kuandika maelekezo wazi, yaliyo na undani kwa wakala wako.

Maelekezo mazuri huainisha:
- **Nani** wakala ni (mhusika na mtindo)
- **Nini** inapaswa kufanya (majukumu hatua kwa hatua)
- **Jinsi** inapaswa tabia yake (vikwazo na mtindo)

Hapo chini, tunaunda wakala wa usaidizi wa kusafiri mwenye maelekezo wazi yanayounda kila majibu anayotoa.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Matokeo Yaliyo Pangwa na Modeli za Pydantic

Maandishi ya aina huru ni muhimu kwa mazungumzo, lakini mifumo inayofuatilia inahitaji data iliyopangwa.
Kwa kuunganisha **modeli za Pydantic** na **kazi ya zana**, tunaweza:

- Kueleza muundo kamili kwa matokeo ya wakala
- Kuhakiki majibu moja kwa moja
- Kuunganisha matokeo ya wakala katika mantiki ya programu kwa uaminifu

Pia tunatambulisha zana inayorejesha maelezo ya mahali pa mwisho ili wakala aweke mapendekezo yake katika data halisi.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Mfumo wa 3: Wakala wa Uwajibikaji Mmoja

Kazi tata hupata faida kwa kugawanya kazi kwa mawakala wengi wenye lengo maalum, kila mmoja akiwa na uwajibikaji mmoja:

- **Mtaalamu wa Mahali** anayejua kuhusu maeneo na upatikanaji
- **Mratibu wa Usafirishaji** anayeshughulikia ndege, hoteli, na ratiba

Hii inafanana na kanuni ya uhandisi wa programu ya *mgawanyo wa majukumu* — kila wakala ni rahisi kujaribiwa, kudumishwa, na kuboreshwa kwa uhuru.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Muhtasari

Katika somo hili tulitumia mifumo mitatu ya muundo wa mawakala katika hali ya mtoaji mapendekezo ya usafiri:

| Mfumo | Wazo kuu | Faida |
|---|---|---|
| **Maelekezo Wazi** | Taja persona, majukumu, na vizingiti mapema | Tabia thabiti, inayoendana na chapa ya wakala |
| **Matokeo Yenye Muundo** | Tumia mifano ya Pydantic kama muundo wa majibu | Matokeo yaliyothibitishwa, yanayoweza kusomwa na mashine |
| **Jukumu Moja** | Mpa kila wakala kazi moja iliyolengwa | Rahisi kupima, kutunza, na kuunda mchanganyiko |

Mifumo hii huunganishwa kwa njia ya asili — unaweza kuunganisha maelekezo wazi na matokeo yenye muundo ndani ya wakala mwenye jukumu moja ili kuunda mifumo imara, tayari kwa uzalishaji.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kiwango cha Hukumu**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kuhakikisha usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upotovu wa maana. Nyaraka ya awali katika lugha yake ya asili inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu ya binadamu inapendekezwa. Hatubebeki jukumu kwa kutoelewana au tafsiri potofu zinazotokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
